### Welcome to Week 6 Day 3!

Let's experiment with a bunch more MCP Servers

In [10]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace, set_tracing_disabled
from agents.mcp import MCPServerStdio
import os
from IPython.display import Markdown, display
from datetime import datetime
load_dotenv(override=True)

if os.getenv("OPENROUTER_API_KEY"):
    key = os.getenv("OPENROUTER_API_KEY", "")
    os.environ["OPENAI_API_KEY"] = key
    base = os.getenv("OPENROUTER_API_BASE", "https://openrouter.ai/api/v1")
    os.environ["OPENAI_BASE_URL"] = base

    set_tracing_disabled(disabled=True)

### The first type of MCP Server: runs locally, everything local

Here's a really interesting one: a knowledge-graph based memory.

It's a persistent memory store of entities, observations about them, and relationships between them.

https://github.com/modelcontextprotocol/servers/tree/main/src/memory


In [2]:
from pathlib import Path

def _memory_libsql_url(db_name: str = "ed.db") -> str:
    """Use 6_mcp/memory when cwd is repo root; use ./memory when cwd is already 6_mcp."""
    cwd = Path.cwd()
    mem_dir = (cwd / "6_mcp" / "memory") if (cwd / "6_mcp").is_dir() else (cwd / "memory")
    mem_dir.mkdir(parents=True, exist_ok=True)
    return f"file:{(mem_dir / db_name).resolve().as_posix()}"

params = {
    "command": "npx",
    "args": ["-y", "mcp-memory-libsql"],
    "env": {"LIBSQL_URL": _memory_libsql_url("ed.db")},
}

async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as server:
    mcp_tools = await server.list_tools()

mcp_tools

[Tool(name='create_entities', title='Create new entities with observations', description='Create new entities with observations', inputSchema={'type': 'object', 'properties': {'entities': {'type': 'array', 'items': {'type': 'object', 'properties': {'name': {'type': 'string'}, 'entityType': {'type': 'string'}, 'observations': {'type': 'array', 'items': {'type': 'string'}}}, 'required': ['name', 'entityType', 'observations']}}}, 'required': ['entities'], '$schema': 'http://json-schema.org/draft-07/schema#'}, outputSchema=None, icons=None, annotations=None, meta=None),
 Tool(name='search_nodes', title='Search for entities and their relations using text search with relevance ranking', description='Search for entities and their relations using text search with relevance ranking', inputSchema={'type': 'object', 'properties': {'query': {'type': 'string'}, 'limit': {'type': 'number'}}, 'required': ['query'], '$schema': 'http://json-schema.org/draft-07/schema#'}, outputSchema=None, icons=None, 

In [3]:
instructions = """You have MCP knowledge-graph memory tools: create_entities, search_nodes, read_graph, create_relations.

1. When the user shares facts about themselves, call create_entities (and create_relations when it helps) before you finish your reply so facts persist in the graph.
2. When the user asks what you know or remember about them, call search_nodes first (use their name or relevant keywords). If that returns little or nothing, try read_graph. Never claim you have no stored information until you have run at least one of those tools and seen the result.
3. Answer from tool results; only fall back to the current user message for information that is explicitly in this turn but not in the graph."""
request = "My name's Ed. I'm an LLM engineer. I'm teaching a course about AI Agents, including the incredible MCP protocol. \
MCP is a protocol for connecting agents with tools, resources and prompt templates, and makes it easy to integrate AI agents with capabilities."
memory_recall_prompt = "My name's Ed. What do you know about me?"
model = "gpt-4.1-mini"

In [4]:
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

Thanks for sharing, Ed! I've noted that you are an LLM engineer teaching a course about AI Agents, and that you highlighted the MCP protocol as a key topic — a protocol that connects agents with tools, resources, and prompt templates, making integration of AI agents with capabilities easier.

If you want, I can keep track of more details or help you retrieve this info anytime. Would you like me to add anything else about your AI Agents course or MCP?

In [5]:
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, memory_recall_prompt)
    display(Markdown(result.final_output))

I know that you are Ed, an LLM engineer, and you are teaching a course about AI Agents. Is there anything else you would like me to remember about you?

### Check the trace:

https://platform.openai.com/traces

### The 2nd type of MCP server - runs locally, calls a web service

### Web search with Serper (Google results API)

Instead of Brave Search, use [Serper](https://serper.dev): create an API key (they include free monthly searches) and add `SERPER_API_KEY` to your `.env`.

This lab runs the [serper-mcp-server](https://pypi.org/project/serper-mcp-server/) package via **`uvx`**. Install [uv](https://docs.astral.sh/uv/getting-started/installation/) if needed; the first run may take a moment while `uvx` downloads the server.

In [6]:
params = {
    "command": "uvx",
    "args": ["serper-mcp-server"],
    "env": {"SERPER_API_KEY": os.getenv("SERPER_API_KEY")},
}

async with MCPServerStdio(params=params, client_session_timeout_seconds=60) as server:
    mcp_tools = await server.list_tools()

mcp_tools

[Tool(name='google_search', title=None, description='Search Google for results', inputSchema={'properties': {'q': {'description': 'The query to search for', 'title': 'Q', 'type': 'string'}, 'gl': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None, 'description': 'The country to search in, e.g. us, uk, ca, au, etc.', 'title': 'Gl'}, 'location': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None, 'description': 'The location to search in, e.g. San Francisco, CA, USA', 'title': 'Location'}, 'hl': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None, 'description': 'The language to search in, e.g. en, es, fr, de, etc.', 'title': 'Hl'}, 'page': {'anyOf': [{'pattern': '^[1-9]\\d*$', 'type': 'string'}, {'type': 'null'}], 'default': '1', 'description': 'The page number to return, first page is 1 (integer value as string)', 'title': 'Page'}, 'tbs': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None, 'description': 'The time period to sea

In [7]:
instructions = "You are able to search the web for information and briefly summarize the takeaways."
request = f"Please research the latest news on Amazon stock price and briefly summarize its outlook. \
For context, the current date is {datetime.now().strftime('%Y-%m-%d')}"
model = "gpt-4o-mini"

In [9]:
async with MCPServerStdio(params=params, client_session_timeout_seconds=60) as mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

Here are the latest updates on Amazon's stock price as of March 25, 2026:

1. **Current Stock Performance**: On March 24, 2026, Amazon's stock (AMZN) closed at $207.24, having dropped by 1.4%. It has been reported that the stock price is roughly 20% lower than its peaks earlier in the year. Analysts have noted pressures from various sectors, particularly concerns related to Amazon Web Services (AWS) and AI spending.

2. **Market Sentiment**: Despite a slight recovery in stock price, analysts are cautious about future performance. For instance, an article highlighted a prediction that Amazon's stock could potentially drop to as low as $150 based on key technical indicators. This has raised concerns among investors.

3. **Analyst Perspectives**: Some analysts believe that Amazon's downturn could represent a buying opportunity, as it has been one of the worst-performing stocks among major tech players in recent months, specifically mentioning a 7% decline in 2026 so far. Another perspective suggests that while the stock has faced challenges, the rebound could be substantial if market conditions change positively.

4. **Future Outlook**: A forecast from a recent analysis suggests uncertainty with potential for either a rebound or prolonged bearish pressure throughout 2026. Some articles reference that Wall Street is reassessing the implications of Amazon's significant investments in AI, which could positively impact future earnings if successful.

For further details, you can refer to these articles:
- [Amazon Stock Quote Price and Forecast (CNN)](https://www.cnn.com/markets/stocks/AMZN)
- [Prediction: This Will Be Amazon's Stock Price in 5 Years (The Motley Fool)](https://www.fool.com/investing/2026/03/20/prediction-this-will-be-amazons-stock-price-in-5-y/)
- [Amazon Stock Forecast and Updates (Capital.com)](https://capital.com/en-int/market-updates/amazon-stock-forecast-19-03-2026)

Overall, the outlook for Amazon's stock is cautious, with concerns surrounding its current price levels and potential future declines tempered by discussions of buying opportunities and the potential for recovery.

### As usual, check out the trace:

https://platform.openai.com/traces

## And now the third type: running remotely

It's actually really hard to find a "remote MCP server" aka "hosted MCP server" aka "managed MCP server".

It's not a common model for using or sharing MCP servers, and there isn't a standard way to discover remote MCP servers.

Anthropic lists some remote MCP servers, but these are for paid applications with business users:

https://docs.anthropic.com/en/docs/agents-and-tools/remote-mcp-servers

CloudFlare has tooling for you to create and deploy your own remote MCP servers, but this does not seem to be a common practice:

https://developers.cloudflare.com/agents/guides/remote-mcp-server/


# And back to the 2nd type: the Polygon.io MCP Server

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">PLEASE READ!!-</h2>
            <span style="color:#ff7800;">This service for financial market data has both a FREE plan and a PAID plan, and we can use either depending on your appetite.
            </span>
        </td>
    </tr>
</table>

## NEW SECTION: Introducing polygon.io

Polygon.io is a hugely popular financial data provider. It has a free plan and a paid plan. And it also has an MCP Server!

First, read up on polygon.io on their excellent website, including looking at their pricing:

https://polygon.io

### Polygon.io Part 1: Polygon.io free service (the paid will be totally optional, of course!)

1. Please sign up for polygon.io (top right)  
2. Once signed in, please select "Keys" in the left hand navigation
3. Press the blue "New Key" button
4. Copy the key name
5. Edit your .env file and add the row:

`POLYGON_API_KEY=xxxx`

In [11]:
load_dotenv(override=True)
polygon_api_key = os.getenv("POLYGON_API_KEY")
if not polygon_api_key:
    print("POLYGON_API_KEY is not set")

In [12]:
from polygon import RESTClient
client = RESTClient(polygon_api_key)
client.get_previous_close_agg("AAPL")[0]

PreviousCloseAgg(ticker='AAPL', close=251.64, high=254.825, low=249.55, open=250.35, timestamp=1774382400000, volume=45152288.0, vwap=252.1598)

### Wrapped into a python module that caches end of day prices

I've made a python module `market.py` that uses this API to look up share prices.

But the free API is quite heavily rate limited - so I've been a bit sneaky; when you ask for a share price, this function retrieves the entire end-of-day equity market, and caches it in our database.


In [13]:
from market import get_share_price
get_share_price("AAPL")

251.64

In [14]:
# no rate limiting concerns!

for i in range(1000):
    get_share_price("AAPL")
get_share_price("AAPL")

251.64

### And I've made this into an MCP Server

Just as we did with accounts.py; see `market_server.py`

In [15]:
params = {"command": "uv", "args": ["run", "market_server.py"]}
async with MCPServerStdio(params=params, client_session_timeout_seconds=60) as server:
    mcp_tools = await server.list_tools()
mcp_tools

[Tool(name='lookup_share_price', title=None, description='This tool provides the current price of the given stock symbol.\n\n    Args:\n        symbol: the symbol of the stock\n    ', inputSchema={'properties': {'symbol': {'title': 'Symbol', 'type': 'string'}}, 'required': ['symbol'], 'title': 'lookup_share_priceArguments', 'type': 'object'}, outputSchema={'properties': {'result': {'title': 'Result', 'type': 'number'}}, 'required': ['result'], 'title': 'lookup_share_priceOutput', 'type': 'object'}, icons=None, annotations=None, meta=None)]

### Let's try it out!

Hopefully gpt-4o-mini is smart enough to know that the symbol for Apple is AAPL

In [16]:
instructions = "You answer questions about the stock market."
request = "What's the share price of Apple?"
model = "gpt-4.1-mini"

async with MCPServerStdio(params=params, client_session_timeout_seconds=60) as mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

The current share price of Apple (AAPL) is $251.64.

## Polygon.io Part 2: Paid Plan - Totally Optional!

If you are interested, you can subscribe to the monthly plan to get more up to date market data, and unlimited API calls.

If you do wish to do this, then it also makes sense to use the full MCP server that Polygon.io has released, to take advantage of all their functionality.



In [ ]:

params = {"command": "uvx",
          "args": ["--from", "git+https://github.com/polygon-io/mcp_polygon@v0.1.0", "mcp_polygon"],
          "env": {"POLYGON_API_KEY": polygon_api_key}
          }
async with MCPServerStdio(params=params, client_session_timeout_seconds=60) as server:
    mcp_tools = await server.list_tools()
mcp_tools


### Wow that's a lot of tools!

Let's try them out - hopefully the sheer number of tools doesn't overwhelm gpt-4o-mini!

With the $29 monthly plan, we don't have access to some of the APIs, so I've needed to specify which APIs can be called.

If you've splashed out on a bigger plan, feel free to remove my extra constraint..

In [ ]:
instructions = "You answer questions about the stock market."
request = "What's the share price of Apple? Use your get_snapshot_ticker tool to get the latest price."
model = "gpt-4.1-mini"

async with MCPServerStdio(params=params, client_session_timeout_seconds=60) as mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

## Setting up your .env file

If you do decide to have a paid plan, please add this to your .env file to indicate:

`POLYGON_PLAN=paid`

And if you decide to go all the way for the realtime API, then please do:

`POLYGON_PLAN=realtime`

In [ ]:
load_dotenv(override=True)

polygon_plan = os.getenv("POLYGON_PLAN")
is_paid_polygon = polygon_plan == "paid"
is_realtime_polygon = polygon_plan == "realtime"

if is_paid_polygon:
    print("You've chosen to subscribe to the paid Polygon plan, so the code will look at prices on a 15 min delay")
elif is_realtime_polygon:
    print("Wowzer - you've chosen to subscribe to the realtime Polygon plan, so the code will look at realtime prices")
else:
    print("According to your .env file, you've chosen to subscribe to the free Polygon plan, so the code will look at EOD prices")

## And that's it for today!

I've removed the part of this lab that uses the "Financial Datasets" mcp server, because it's inferior - more expensive with fewer APIs.

And this way we get to use the same provider for Free and Paid APIs.

But if you want to see the code, just look in the git history for a prior version.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercises</h2>
            <span style="color:#ff7800;">Explore MCP server marketplaces and integrate your own, using all 3 approaches.
            </span>
        </td>
    </tr>
</table>